# Model evaluation

This notebook uses linear regression as a baseline, then compares several other regression models on the concrete dataset. The final section shows Poisson regression on count data.

1. **Linear regression**
2. **K-nearest neighbors (KNN) regression**
3. **Decision tree regression**
4. **Random forest regression**
5. **Optimized random forest regression**
6. **Support vector regression (SVR)**
7. **Gaussian process regression (GPR)**
8. **Model comparison**

## Notebook setup

### Import libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

: 

### Dataset: concrete compressive strength

The [UCI concrete compressive strength dataset](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength) has 1,030 mix designs with ingredient amounts, curing age, and measured strength in MPa.

In [ ]:
# Load from OpenML (cached after first download)
raw = fetch_openml(data_id=4353, as_frame=True, parser='auto')
df  = raw.frame.copy()

df.columns = [
    'cement', 'slag', 'fly_ash', 'water', 'superplasticizer',
    'coarse_agg', 'fine_agg', 'age', 'strength'
]

df.to_csv('../data/concrete_data.csv', index=False)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Many mixes don't use every ingredient
zero_counts = (df[['slag', 'fly_ash', 'superplasticizer']] == 0).sum()
print('Zero counts (ingredients not present in every mix):')
print(zero_counts.to_string())
print()
df.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].hist(df['strength'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Compressive strength (MPa)')
axes[0].set_title('Target distribution')

axes[1].scatter(df['age'], df['strength'], s=8, alpha=0.3, color='teal')
axes[1].set_xlabel('Curing age (days)')
axes[1].set_ylabel('Strength (MPa)')
axes[1].set_title('Strength vs curing age')

axes[2].scatter(df['cement'], df['strength'], s=8, alpha=0.3, color='purple')
axes[2].set_xlabel('Cement content (kg/m³)')
axes[2].set_ylabel('Strength (MPa)')
axes[2].set_title('Strength vs cement content')

plt.tight_layout()
plt.show()

In [ ]:
feature_names = [c for c in df.columns if c != 'strength']
X = df[feature_names].values
y = df['strength'].values

print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Target range: {y.min():.1f} to {y.max():.1f} MPa')
print(f'Mean strength: {y.mean():.1f} MPa')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=315)
print(f'\nTraining samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

results = {}

## 1. Linear regression

Linear regression fits a weighted sum of the inputs plus an intercept.

### Prediction equation

$$
\hat{y} = \mathbf{w}^\top \mathbf{x} + b
$$

### What is learned during training

- One coefficient for each feature in $\mathbf{w}$
- One intercept value $b$

### How to calculate one prediction

1. Take the feature row $\mathbf{x}$ for the new sample.
2. Multiply each feature by its learned coefficient and sum.
3. Add the learned intercept $b$.
4. Return the result as the prediction.

Use it as a baseline when you want a fast, interpretable first pass.

**Strengths:**
- Fast
- Easy to interpret
- Good baseline

**Weaknesses:**
- Limited for non-linear patterns
- Sensitive to outliers
- Weak on interactions without feature engineering

**Important hyperparameters:**
- `fit_intercept`: include an intercept term
- `copy_X`: keep a copy of the training matrix
- `n_jobs`: parallelization for some solver paths

**Preprocessing:** not required for fitting, though scaling can help if you want to compare coefficient sizes.

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

y_pred_lin = lin_model.predict(X_test)
lin_rmse   = root_mean_squared_error(y_test, y_pred_lin)
lin_r2     = r2_score(y_test, y_pred_lin)

print(f'Linear regression RMSE: {lin_rmse:.2f} MPa')
print(f'Linear regression R²:   {lin_r2:.3f}')
results['Linear regression'] = {'rmse': lin_rmse, 'r2': lin_r2}

## 2. K-nearest neighbors (KNN) regression

KNN predicts from the average target value of the nearest training points.

### Prediction equation

$$
\hat{y}(x)=\frac{\sum_{i \in \mathcal{N}_k(x)} w_i\,y_i}{\sum_{i \in \mathcal{N}_k(x)} w_i}
$$

### What is learned during training

- KNN is mostly instance-based: it stores the training set
- It uses the chosen distance metric and neighbor settings
- With distance weighting, it uses larger weights for closer neighbors

### How to calculate one prediction

1. Scale the new sample using the fitted scaler.
2. Compute distances from the new sample to all training samples.
3. Select the $k$ closest neighbors.
4. Compute neighbor weights from distance (or uniform weights).
5. Return the weighted average of neighbor target values.

**Strengths:**
- Simple
- Local and flexible
- Strong non-parametric baseline

**Weaknesses:**
- Slow at prediction time
- Sensitive to scale
- Struggles in high dimensions

**Important hyperparameters:**
- `n_neighbors`: number of neighbors to average
- `weights`: use uniform or distance weighting
- `metric`: distance function used for neighbor search

**Preprocessing:** scaling is essential.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsRegressor(n_neighbors=5, weights='distance'))
])
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)
knn_rmse   = root_mean_squared_error(y_test, y_pred_knn)
knn_r2     = r2_score(y_test, y_pred_knn)

print(f'KNN RMSE: {knn_rmse:.2f} MPa')
print(f'KNN R²:   {knn_r2:.3f}')
results['KNN'] = {'rmse': knn_rmse, 'r2': knn_r2}

## 3. Decision tree regression

A decision tree splits the feature space into regions and predicts the mean target value in each leaf.

### Prediction equation

$$
\hat{y}(x)=c_{m(x)}
$$

Here, $m(x)$ is the leaf reached by $x$, and $c_{m(x)}$ is that leaf's learned value.

### What is learned during training

- Which feature to split on at each node
- The threshold value for each split
- The leaf prediction value (typically mean target in that leaf)

### How to calculate one prediction

1. Start at the root node.
2. Compare the relevant feature to the node threshold.
3. Move left or right based on the comparison.
4. Repeat until reaching a leaf.
5. Return that leaf's prediction value.

**Strengths:**
- No scaling needed
- Captures non-linear patterns
- Easy to visualize

**Weaknesses:**
- Can overfit
- Produces step-like predictions
- Poor extrapolation

**Important hyperparameters:**
- `max_depth`: limit tree depth
- `min_samples_leaf`: keep leaves from getting too small
- `min_samples_split`: require enough samples before splitting
- `max_features`: limit features considered at each split

**Preprocessing:** none required.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(max_depth=5, random_state=315)
result = dt_model.fit(X_train, y_train)

y_pred_test_dt = dt_model.predict(X_test)
dt_test_rmse   = root_mean_squared_error(y_test, y_pred_test_dt)
dt_test_r2     = r2_score(y_test, y_pred_test_dt)

print(f'Decision tree RMSE: {dt_test_rmse:.2f} MPa')
print(f'Decision tree R²:   {dt_test_r2:.3f}')
results['Decision tree'] = {'rmse': dt_test_rmse, 'r2': dt_test_r2}

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(12, 6))
plot_tree(dt_model, max_depth=2, feature_names=feature_names, fontsize=10, filled=True, rounded=True)
plt.show()

## 4. Random forest regression

A random forest averages many decision trees to reduce variance and improve stability.

### Prediction equation

$$
\hat{y}(x)=\frac{1}{T}\sum_{t=1}^{T} f_t(x)
$$

where $f_t(x)$ is the prediction from tree $t$, and $T$ is the number of trees.

### What is learned during training

- The full structure of each tree: splits, thresholds, and leaf values
- The ensemble average behavior across all trained trees

### How to calculate one prediction

1. Send the new sample through each tree.
2. Get one prediction from each tree.
3. Average all tree predictions.
4. Return that average as the forest prediction.

**Strengths:**
- Usually stronger than a single tree
- Handles non-linear patterns well
- Gives feature importance estimates

**Weaknesses:**
- Less interpretable
- Uses more memory
- Still cannot extrapolate well

**Important hyperparameters:**
- `n_estimators`: number of trees
- `max_depth`: maximum depth of each tree
- `min_samples_leaf`: minimum samples per leaf
- `max_features`: number of features checked at each split

**Preprocessing:** none required.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=315,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_rmse   = root_mean_squared_error(y_test, y_pred_rf)
rf_r2     = r2_score(y_test, y_pred_rf)

print(f'Random forest  RMSE: {rf_rmse:.2f} MPa')
print(f'Random forest  R²:   {rf_r2:.3f}')
results['Random forest'] = {'rmse': rf_rmse, 'r2': rf_r2}

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values()
importances.plot(kind='barh', color='steelblue', title='Feature importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Optimized random forest regression

Here we tune a random forest with cross-validation and keep the best setting. A small search for optimal settings can often beat a hand-picked defaults with little extra complexity.

**Important hyperparameters:**
- `n_estimators`: number of trees to search over
- `max_depth`: tree depth values to test
- `min_samples_leaf`: leaf size values to test
- `cv`: number of cross-validation folds

In [ ]:
%%time

rf_param_grid = {
    'n_estimators': [200, 400, 500, 600],
    'max_depth': [4, 6, 8, 10, 12, 14, 16, 18],
    'min_samples_leaf': [1, 2, 3, 4]
}

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=315, n_jobs=-1),
    rf_param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

opt_rf_model = rf_search.best_estimator_
y_pred_opt_rf = opt_rf_model.predict(X_test)
opt_rf_rmse = root_mean_squared_error(y_test, y_pred_opt_rf)
opt_rf_r2 = r2_score(y_test, y_pred_opt_rf)

print(f'Best parameters: {rf_search.best_params_}')
print(f'Optimized random forest RMSE: {opt_rf_rmse:.2f} MPa')
print(f'Optimized random forest R²:   {opt_rf_r2:.3f}')
results['Optimized random forest'] = {'rmse': opt_rf_rmse, 'r2': opt_rf_r2}
print()

## 7. Support vector regression (SVR)

SVR predicts with a smooth curve while ignoring small errors inside an `epsilon` tube. A simple way to think about it:

- For a given observation, the kernel value tell us the other points that are "nearby".
- Points that are near each other should have similar values.
- The 'difficult' points that are far away from the rest exert the most influence on predictions.

### Prediction equation

For a new input $x$, SVR prediction is:
$$
\hat{y}(x)=\sum_{i \in SV} w_i\,K(x_i, x)+b
$$

Where:
- $x_i$: support vectors from training
- $K(x_i, x)$: kernel similarity between support vector $x_i$ and new point $x$
- $w_i$: learned weights
- $b$: intercept

### How to calculate one prediction

1. Scale the new input using the fitted scaler.
2. Compute kernel similarity from the new input to each support vector.
3. Multiply each similarity value by its learned weight.
4. Sum these weighted similarities.
5. Add the intercept $b$.
6. Return the result as the predicted value.

**Strengths:**
- Works well with a good kernel
- Handles non-linear patterns
- Robust to some outliers

**Weaknesses:**
- Needs scaling
- Sensitive to hyperparameters
- Can be slow on larger datasets

**Important hyperparameters:**
- `C`: controls regularization strength
- `epsilon`: width of the no-penalty tube
- `kernel`: shape of the regression function
- `gamma`: kernel width for RBF kernels

**Preprocessing:** scaling is mandatory.

In [ ]:
from sklearn.svm import SVR

svr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svr',    SVR(kernel='rbf', C=100, epsilon=1.0))
])
svr_model.fit(X_train, y_train)

y_pred_svr = svr_model.predict(X_test)
svr_rmse   = root_mean_squared_error(y_test, y_pred_svr)
svr_r2     = r2_score(y_test, y_pred_svr)

print(f'SVR RMSE: {svr_rmse:.2f} MPa')
print(f'SVR R²:   {svr_r2:.3f}')
results['SVR'] = {'rmse': svr_rmse, 'r2': svr_r2}

## 8. Gaussian process regression (GPR)

GPR predicts a full distribution at each point, not just one number.

In plain language:
- The kernel quantifies how strongly each training point should influence another point.
- The model combines those influences to get a mean prediction.
- It also returns uncertainty, which usually grows in regions with less similar training data.

### Inference equation

For a new point $x_*$, GPR computes the mean prediction as:
$$
\mu_* = K(x_*, X)\,[K(X, X)+\sigma_n^2 I]^{-1} y
$$

Where:
- $\mu_*$ is the predicted mean
- $K(\cdot,\cdot)$ the chosen kernel
- $X, y$ are training inputs and targets
- $\sigma_n^2$ is noise level, captured by `WhiteKernel` or `alpha`
- $I$ is the identity matrix

The same covariance terms can also be used to calculate the variability, or uncertainty, around that prediction.

### What is learned during training

The free parameters in the kernel function are optimized so that the output from the model aligns with the training data. In the example RBF kernel function above, the gamma parameter would be optimized. In the GPR example below, the main learned parameters are the Matérn length scale and the `WhiteKernel` noise level. Scikit-learn chooses them by maximizing the log marginal likelihood of the training data.

The log marginal likelihood is the log probability of observing the training data values with the current kernel settings. It tells us how well the kernel explains the data. Scikit-learn calculates it from the covariance matrix and uses that score to decide which kernel parameters make the data most probable.

The code cell below plots the log marginal likelihood as a function of length scale for one feature, so you can see the optimization idea directly.

In [ ]:
# Visualize how the training likelihood changes as the length scale changes.
# This one-feature demo keeps the picture simple while using the learned noise level from the full model.
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

age_feature_index = feature_names.index('age')
X_train_gpr_demo = X_train_scaled[:, [age_feature_index]]

learned_noise_level = np.float64(0.04063645443470903)
gpr_demo_kernel = Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=1.5) + WhiteKernel(
    noise_level=learned_noise_level,
    noise_level_bounds='fixed'
)

gpr_demo = GaussianProcessRegressor(
    kernel=gpr_demo_kernel,
    optimizer=None,
    normalize_y=True,
)
gpr_demo.fit(X_train_gpr_demo, y_train)

length_scales = np.logspace(-2, 2, 200)
log_marginal_likelihood = np.array([
    gpr_demo.log_marginal_likelihood(theta=np.log([length_scale]))
    for length_scale in length_scales
])

best_idx = np.argmax(log_marginal_likelihood)
best_length_scale = length_scales[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(length_scales, log_marginal_likelihood, color='teal', linewidth=2)
ax.axvline(best_length_scale, color='black', linestyle='--', linewidth=1)
ax.scatter([best_length_scale], [log_marginal_likelihood[best_idx]], color='crimson', zorder=3)
ax.set_xlabel('Length scale')
ax.set_ylabel('Log marginal likelihood')
ax.set_title(f'Likelihood vs length scale for {feature_names[age_feature_index]}')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Best length scale in this demo: {best_length_scale:.3f}')
print(f'Best log marginal likelihood:   {log_marginal_likelihood[best_idx]:.3f}')

### How to calculate one prediction

1. Scale the new input with the fitted scaler.
2. Compute kernel similarities between the new point and all training points.
3. Use the trained covariance term $[K(X, X)+\sigma_n^2 I]^{-1}$.
4. Combine those learned covariance terms to get the mean prediction.
5. Compute $\sigma_*^2$ to get uncertainty at that point.
6. Report prediction as mean plus uncertainty (for example, $\mu_* \pm 1.96\sigma_*$).

**Strengths:**
- Gives uncertainty estimates
- Strong on small datasets
- Flexible Bayesian framework

**Weaknesses:**
- Expensive to train
- Kernel choice matters
- Does not scale well

**Important hyperparameters:**
- `length_scale`: width of the kernel
- `kernel`: function prior, such as Matérn or RBF
- `n_restarts_optimizer`: extra kernel optimization attempts
- `alpha`: stability term added to the covariance matrix
- `WhiteKernel`: learns the noise level in the data

**Preprocessing:** scaling is recommended.

In [ ]:
scaler_gpr     = StandardScaler()
X_train_scaled = scaler_gpr.fit_transform(X_train)
X_test_scaled  = scaler_gpr.transform(X_test)

kernel = 1.0 * Matern(length_scale=np.ones(X_train.shape[1]), length_scale_bounds=(1e-2, 1e2), nu=1.5) + WhiteKernel(noise_level=1.0)

gpr_model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    random_state=315,
    normalize_y=True
)

gpr_model.fit(X_train_scaled, y_train)

y_pred_gpr, y_std_gpr = gpr_model.predict(X_test_scaled, return_std=True)
gpr_rmse = root_mean_squared_error(y_test, y_pred_gpr)
gpr_r2   = r2_score(y_test, y_pred_gpr)

print(f'           GPR RMSE: {gpr_rmse:.2f} MPa')
print(f'             GPR R²: {gpr_r2:.3f}')
print(f'Mean prediction std: {y_std_gpr.mean():.2f} MPa')
results['GPR'] = {'rmse': gpr_rmse, 'r2': gpr_r2}

## 9. Model comparison

In [ ]:
names = list(results.keys())
rmses = [results[n]['rmse'] for n in names]
r2s   = [results[n]['r2']   for n in names]

order   = np.argsort(rmses)
names_s = [names[i] for i in order]
rmses_s = [rmses[i] for i in order]
r2s_s   = [r2s[i]   for i in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.barh(names_s, rmses_s, color='steelblue')
ax1.set_xlabel('Test RMSE (MPa) - lower is better')
ax1.set_title('Test RMSE by model')

ax2.barh(names_s, r2s_s, color='teal')
ax2.set_xlabel('Test R² - higher is better')
ax2.set_title('Test R² by model')
ax2.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Concrete compressive strength', fontsize=16)
plt.tight_layout()
plt.show()

print(f'{"Model":<23} {"RMSE (MPa)":>12} {"R²":>8}')
print('-' * 45)

for n, r, r2 in zip(names_s, rmses_s, r2s_s):
    print(f'{n:<23} {r:>12.2f} {r2:>8.3f}')


## 11. SKLearn pipeline optimization

Scikit-learn's [`Pipeline()`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) lets you chain together preprocessing steps and a model into a single model-like object. This is great for optimization and deployment because you don't have to manage each individual step in the preprocessing workflow. Downside - it only works with sklearn classes. If you are doing custom feature engineering, you will still need to do those steps outside of the pipeline (or wrap them in a sklearn-like class).

### 11.1. PolyFeatures -> PCA -> SCV

**Note:** I don't necessarily recommend this combination of preprocessing and model - it's just an example of how to optimize a pipeline with multiple steps and hyperparameters.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures

# Pipeline: PolynomialFeatures > PCA > StandardScaler > SVM
pipeline = Pipeline(
    [
        ('poly', PolynomialFeatures()),
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('svr', SVR())
    ]
)

In [ ]:
%%time
# Specify the parameter search space as a list of
# dictionaries, one for each polynomial degree to test.
# This allows us to test different PCA component counts for each degree.
params = [
    {
        'poly__degree': [1], # Degree 1 gives 8 features
        'pca__n_components': [5,6,7],
        'svr__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'svr__C': [0.25, 0.50, 1.0, 2.0],
        'svr__epsilon': [0.025, 0.05, 0.1, 0.2, 0.4]
    },
    {
        'poly__degree': [2], # Degree 2 gives 46 features
        'pca__n_components': [16, 36, 40, 45], #list(range(4,46)),
        'svr__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'svr__C': [0.25, 0.50, 1.0, 2.0],
        'svr__epsilon': [0.025, 0.05, 0.1, 0.2, 0.4]
    },
    {
        'poly__degree': [3], # Degree 3 gives 165 features
        'pca__n_components': [32, 64, 128], #list(range(4,166)),
        'svr__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'svr__C': [0.25, 0.50, 1.0, 2.0],
        'svr__epsilon': [0.025, 0.05, 0.1, 0.2, 0.4]
    }
]

grid_search = GridSearchCV(
    pipeline,
    params,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
best_params = grid_search.best_params_

for key, val in best_params.items():
    print(f'{key}: {val}')

In [ ]:
winner = grid_search.best_estimator_

y_pred_svr = winner.predict(X_test)
svr_rmse   = root_mean_squared_error(y_test, y_pred_svr)
svr_r2     = r2_score(y_test, y_pred_svr)

print(f'Optimized SVR pipeline RMSE: {svr_rmse:.2f} MPa')
print(f'Optimized SVR pipeline R²:   {svr_r2:.3f}')
results['Optimized SVR pipeline'] = {'rmse': svr_rmse, 'r2': svr_r2}

In [ ]:
names = list(results.keys())
rmses = [results[n]['rmse'] for n in names]
r2s   = [results[n]['r2']   for n in names]

order   = np.argsort(rmses)
names_s = [names[i] for i in order]
rmses_s = [rmses[i] for i in order]
r2s_s   = [r2s[i]   for i in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.barh(names_s, rmses_s, color='steelblue')
ax1.set_xlabel('Test RMSE (MPa) - lower is better')
ax1.set_title('Test RMSE by model')

ax2.barh(names_s, r2s_s, color='teal')
ax2.set_xlabel('Test R² - higher is better')
ax2.set_title('Test R² by model')
ax2.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Concrete compressive strength', fontsize=16)
plt.tight_layout()
plt.show()

print(f'{"Model":<23} {"RMSE (MPa)":>12} {"R²":>8}')
print('-' * 45)

for n, r, r2 in zip(names_s, rmses_s, r2s_s):
    print(f'{n:<23} {r:>12.2f} {r2:>8.3f}')

### 11.2. PolyFeatures -> PCA -> LinearRegression

In [ ]:
pipeline = Pipeline(
    [
        ('poly', PolynomialFeatures()),
        ('scaler', StandardScaler()),
        ('pca', PCA()),
        ('linear', LinearRegression())
    ]
)

In [ ]:
%%time
# Specify the parameter search space as a list of
# dictionaries, one for each polynomial degree to test.
# This allows us to test different PCA component counts for each degree.

params = [
    {
        'poly__degree': [1], # Degree 1 gives 8 features
        'pca__n_components': [4,5,6,7,8],
    },
    {
        'poly__degree': [2], # Degree 2 gives 46 features
        'pca__n_components': list(range(15,46)),
    },
    {
        'poly__degree': [3], # Degree 3 gives 165 features
        'pca__n_components': list(range(105,166,2)),
    }
]

grid_search = GridSearchCV(
    pipeline,
    params,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
best_params = grid_search.best_params_

for key, val in best_params.items():
    print(f'{key}: {val}')

In [ ]:
winner = grid_search.best_estimator_

y_pred_svr = winner.predict(X_test)
svr_rmse   = root_mean_squared_error(y_test, y_pred_svr)
svr_r2     = r2_score(y_test, y_pred_svr)

print(f'Optimized linear pipeline RMSE: {svr_rmse:.2f} MPa')
print(f'Optimized linear pipeline R²:   {svr_r2:.3f}')
results['Optimized linear pipeline'] = {'rmse': svr_rmse, 'r2': svr_r2}

In [ ]:
names = list(results.keys())
rmses = [results[n]['rmse'] for n in names]
r2s   = [results[n]['r2']   for n in names]

order   = np.argsort(rmses)
names_s = [names[i] for i in order]
rmses_s = [rmses[i] for i in order]
r2s_s   = [r2s[i]   for i in order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.barh(names_s, rmses_s, color='steelblue')
ax1.set_xlabel('Test RMSE (MPa) - lower is better')
ax1.set_title('Test RMSE by model')

ax2.barh(names_s, r2s_s, color='teal')
ax2.set_xlabel('Test R² - higher is better')
ax2.set_title('Test R² by model')
ax2.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Concrete compressive strength', fontsize=16)
plt.tight_layout()
plt.savefig('./figures/01-model-evaluation.png', dpi=300)
plt.show()

print(f'{"Model":<25} {"RMSE (MPa)":>12} {"R²":>8}')
print('-' * 45)

for n, r, r2 in zip(names_s, rmses_s, r2s_s):
    print(f'{n:<25} {r:>12.2f} {r2:>8.3f}')